## Day-4-Pandas-2

### mt1.py

In [4]:
import pandas as pd
import numpy as np

np.random.seed(42)

sales_df = pd.DataFrame({
    "Date": pd.date_range("2025-01-01", periods=50, freq="D"),
    "Salesperson": np.random.choice(
        ["Amit", "Riya", "Karan", "Neha", "Priya"],
        50
    ),
    "Region": np.random.choice(
        ["North", "South", "East", "West"],
        50
    ),
    "Product": np.random.choice(
        ["Laptop", "Mobile", "Tablet", "Monitor"],
        50
    ),
    "Units": np.random.randint(1, 20, 50),
    "UnitPrice": np.random.randint(500, 5000, 50)
})

sales_df["Revenue"] = sales_df["Units"] * sales_df["UnitPrice"]

summary = sales_df.groupby("Region").agg(
    total_revenue=("Revenue", "sum"),
    units_sold=("Units", "sum"),
    average_deal_size=("Revenue", "mean")
).reset_index()

pivot = pd.pivot_table(
    sales_df,
    values="Revenue",
    index="Region",
    columns="Product",
    aggfunc="sum",
    fill_value=0
)

salesperson_rank = (
    sales_df.groupby("Salesperson")["Revenue"]
    .sum()
    .reset_index()
)

salesperson_rank["Rank"] = salesperson_rank["Revenue"].rank(
    method="dense",
    ascending=False
)

salesperson_rank = salesperson_rank.sort_values("Rank")

sales_df.to_csv("sales_clean.csv", index=False)

with pd.ExcelWriter("sales_report.xlsx") as writer:
    sales_df.to_excel(writer, sheet_name="raw", index=False)
    summary.to_excel(writer, sheet_name="summary", index=False)
    pivot.to_excel(writer, sheet_name="pivot")

print("Regional Summary:")
print(summary)

print("\nRevenue by Region and Product:")
print(pivot)

print("\nSalesperson Ranking:")
print(salesperson_rank)

print("\nFiles exported:")
print("sales_clean.csv")
print("sales_report.xlsx")

Regional Summary:
  Region  total_revenue  units_sold  average_deal_size
0   East         153214          51       17023.777778
1  North         187429          63       26775.571429
2  South         514586         187       28588.111111
3   West         404593         138       25287.062500

Revenue by Region and Product:
Product  Laptop  Mobile  Monitor  Tablet
Region                                  
East      57939    1647    31125   62503
North     95907    8113    83409       0
South    237608   24054   124697  128227
West      87505   93079    96374  127635

Salesperson Ranking:
  Salesperson  Revenue  Rank
4        Riya   320840   1.0
2        Neha   303342   2.0
3       Priya   246384   3.0
1       Karan   235168   4.0
0        Amit   154088   5.0

Files exported:
sales_clean.csv
sales_report.xlsx


### mt2.py

In [5]:
import pandas as pd
import seaborn as sns

df = sns.load_dataset("titanic")

df["age"] = df["age"].fillna(df["age"].median())
df["fare"] = df["fare"].fillna(df["fare"].median())

df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df["embark_town"] = df["embark_town"].fillna(df["embark_town"].mode()[0])

df["deck"] = df["deck"].cat.add_categories(["Unknown"])
df["deck"] = df["deck"].fillna("Unknown")

df["FamilySize"] = df["sibsp"] + df["parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

df["AgeGroup"] = pd.cut(
    df["age"],
    bins=[0, 12, 18, 35, 60, 100],
    labels=["Child", "Teen", "YoungAdult", "Adult", "Senior"]
)

df["FareBand"] = pd.qcut(
    df["fare"],
    q=4,
    labels=["Low", "Medium", "High", "VeryHigh"],
    duplicates="drop"
)

survival_table = pd.crosstab(
    [df["AgeGroup"], df["sex"]],
    df["survived"],
    normalize="index"
) * 100

print("Survival by AgeGroup and Sex")
print(survival_table)

class_stats = df.groupby("pclass").agg(
    mean_age=("age", "mean"),
    mean_fare=("fare", "mean"),
    survival_rate=("survived", "mean"),
    passenger_count=("survived", "count")
).reset_index()

class_stats["survival_rate"] *= 100

print("\nClass Statistics")
print(class_stats)

class_info = pd.DataFrame({
    "pclass": [1, 2, 3],
    "label": ["First", "Second", "Third"],
    "deck_level": ["Upper", "Middle", "Lower"]
})

df = pd.merge(df, class_info, on="pclass", how="left")

df.to_csv("titanic_final.csv", index=False)

with pd.ExcelWriter("titanic_report.xlsx") as writer:
    df.to_excel(writer, sheet_name="Data", index=False)
    survival_table.to_excel(writer, sheet_name="SurvivalPivot")
    class_stats.to_excel(writer, sheet_name="ClassStats", index=False)

print("\nFiles exported:")
print("titanic_final.csv")
print("titanic_report.xlsx")

Survival by AgeGroup and Sex
survived                   0           1
AgeGroup   sex                          
Child      female  40.625000   59.375000
           male    43.243243   56.756757
Teen       female  25.000000   75.000000
           male    91.176471    8.823529
YoungAdult female  24.855491   75.144509
           male    83.701657   16.298343
Adult      female  22.857143   77.142857
           male    80.800000   19.200000
Senior     female   0.000000  100.000000
           male    89.473684   10.526316

Class Statistics
   pclass   mean_age  mean_fare  survival_rate  passenger_count
0       1  36.812130  84.154687      62.962963              216
1       2  29.765380  20.662183      47.282609              184
2       3  25.932627  13.675550      24.236253              491

Files exported:
titanic_final.csv
titanic_report.xlsx


### mt3.py

In [6]:
import pandas as pd

df = pd.read_csv("data.csv")

df["CustomerName"] = df["CustomerName"].str.strip().str.title()
df["Description"] = df["Description"].str.strip().str.lower()

df["Tokens"] = df["Description"].str.split()

df["HasDiscount"] = df["Description"].str.contains(
    "discount|offer|sale",
    case=False,
    na=False,
    regex=True
)

df["ProductCode"] = df["Description"].str.extract(r"([A-Z]{2}\d{3})")

words = (
    df["Description"]
    .str.split()
    .explode()
    .value_counts()
)

print("Cleaned Data:")
print(df)

print("\nWord Frequency:")
print(words)

Cleaned Data:
    CustomerName                 Description                           Tokens  \
0    Mihir Rathi  laptop ab123 with discount  [laptop, ab123, with, discount]   
1  Pratham Patel  mobile cd456 special offer  [mobile, cd456, special, offer]   
2     Akash Shah        laptop ab123 on sale        [laptop, ab123, on, sale]   

   HasDiscount ProductCode  
0         True         NaN  
1         True         NaN  
2         True         NaN  

Word Frequency:
Description
laptop      2
ab123       2
with        1
discount    1
mobile      1
cd456       1
special     1
offer       1
on          1
sale        1
Name: count, dtype: int64


### mt4.py

In [7]:
import pandas as pd
import numpy as np

np.random.seed(42)

n = 10000

df = pd.DataFrame({
    "round": range(1, n + 1),
    "die1": np.random.randint(1, 7, n),
    "die2": np.random.randint(1, 7, n)
})

df["total"] = df["die1"] + df["die2"]

stats = df.groupby("total").size().reset_index(name="frequency")
stats["probability"] = stats["frequency"] / n

prob_gt_9 = (df["total"] > 9).mean()

combo_freq = pd.pivot_table(
    df,
    index="die1",
    columns="die2",
    values="round",
    aggfunc="count",
    fill_value=0
)

print("Frequency and Probability by Total:")
print(stats)

print("\nProbability of Total > 9:")
print(prob_gt_9)

print("\nDie1 vs Die2 Frequency Table:")
print(combo_freq)

with pd.ExcelWriter("dice_simulation.xlsx") as writer:
    df.to_excel(writer, sheet_name="RawData", index=False)
    stats.to_excel(writer, sheet_name="Stats", index=False)
    combo_freq.to_excel(writer, sheet_name="Pivot")

print("\nFile exported: dice_simulation.xlsx")

Frequency and Probability by Total:
    total  frequency  probability
0       2        273       0.0273
1       3        608       0.0608
2       4        811       0.0811
3       5       1101       0.1101
4       6       1383       0.1383
5       7       1616       0.1616
6       8       1414       0.1414
7       9       1105       0.1105
8      10        858       0.0858
9      11        556       0.0556
10     12        275       0.0275

Probability of Total > 9:
0.1689

Die1 vs Die2 Frequency Table:
die2    1    2    3    4    5    6
die1                              
1     273  291  233  288  294  286
2     317  293  264  279  268  271
3     285  253  287  262  280  258
4     296  283  248  279  276  290
5     240  279  287  299  308  276
6     273  297  272  260  280  275

File exported: dice_simulation.xlsx


### t1.py

In [8]:
import seaborn as sns

titanic = sns.load_dataset("titanic")

top_10 = titanic.sort_values(by="fare", ascending=False)[["who", "fare", "class"]].head(10)

print(top_10)

       who      fare  class
679    man  512.3292  First
258  woman  512.3292  First
737    man  512.3292  First
88   woman  263.0000  First
438    man  263.0000  First
341  woman  263.0000  First
27     man  263.0000  First
742  woman  262.3750  First
311  woman  262.3750  First
299  woman  247.5208  First


### t2.py

In [9]:
import seaborn as sns

titanic = sns.load_dataset("titanic")

youngest = titanic.loc[
    titanic.groupby("pclass")["age"].idxmin(),
    ["pclass", "age", "class"]
]

print(youngest)

     pclass   age   class
305       1  0.92   First
755       2  0.67  Second
803       3  0.42   Third


### t3.py

In [10]:
import pandas as pd

df = pd.DataFrame({
    "Student": ["A", "B", "C", "D", "E", "F", "G", "H", "I", "J"],
    "Marks": [95, 88, 95, 76, 88, 92, 76, 85, 92, 80]
})

df["Rank_average"] = df["Marks"].rank(method="average", ascending=False)
df["Rank_min"] = df["Marks"].rank(method="min", ascending=False)
df["Rank_max"] = df["Marks"].rank(method="max", ascending=False)
df["Rank_first"] = df["Marks"].rank(method="first", ascending=False)
df["Rank_dense"] = df["Marks"].rank(method="dense", ascending=False)

print(df.sort_values("Marks", ascending=False))

  Student  Marks  Rank_average  Rank_min  Rank_max  Rank_first  Rank_dense
0       A     95           1.5       1.0       2.0         1.0         1.0
2       C     95           1.5       1.0       2.0         2.0         1.0
8       I     92           3.5       3.0       4.0         4.0         2.0
5       F     92           3.5       3.0       4.0         3.0         2.0
1       B     88           5.5       5.0       6.0         5.0         3.0
4       E     88           5.5       5.0       6.0         6.0         3.0
7       H     85           7.0       7.0       7.0         7.0         4.0
9       J     80           8.0       8.0       8.0         8.0         5.0
3       D     76           9.5       9.0      10.0         9.0         6.0
6       G     76           9.5       9.0      10.0        10.0         6.0


### t4.py

In [11]:
import seaborn as sns

df = sns.load_dataset("titanic")

sex_survival = df.groupby("sex")["survived"].mean() * 100
print("Survival Rate by Sex:")
print(sex_survival)

class_survival = df.groupby("pclass")["survived"].mean() * 100
print("\nSurvival Rate by Class:")
print(class_survival)

group_survival = df.groupby(["sex", "pclass"])["survived"].mean() * 100
print("\nSurvival Rate by Sex and Class:")
print(group_survival)

best_group = group_survival.idxmax()
best_rate = group_survival.max()

print("\nHighest Survival Group:")
print("Group:", best_group)
print("Rate:", best_rate)

Survival Rate by Sex:
sex
female    74.203822
male      18.890815
Name: survived, dtype: float64

Survival Rate by Class:
pclass
1    62.962963
2    47.282609
3    24.236253
Name: survived, dtype: float64

Survival Rate by Sex and Class:
sex     pclass
female  1         96.808511
        2         92.105263
        3         50.000000
male    1         36.885246
        2         15.740741
        3         13.544669
Name: survived, dtype: float64

Highest Survival Group:
Group: ('female', np.int64(1))
Rate: 96.80851063829788


### t5.py

In [12]:
import seaborn as sns

df = sns.load_dataset("titanic")

fare_stats = df.groupby("pclass")["fare"].agg(["mean", "median", "std"])

print(fare_stats)

             mean   median        std
pclass                               
1       84.154687  60.2875  78.380373
2       20.662183  14.2500  13.417399
3       13.675550   8.0500  11.778142


### t6.py

In [13]:
import seaborn as sns

df = sns.load_dataset("titanic")

df["class_median_age"] = df.groupby("pclass")["age"].transform("median")

df["IsOlderThanClassMedian"] = df["age"] > df["class_median_age"]

print(df[["pclass", "age", "class_median_age", "IsOlderThanClassMedian"]].head())

   pclass   age  class_median_age  IsOlderThanClassMedian
0       3  22.0              24.0                   False
1       1  38.0              37.0                    True
2       3  26.0              24.0                    True
3       1  35.0              37.0                   False
4       3  35.0              24.0                    True


### t7.py

In [14]:
import seaborn as sns

df = sns.load_dataset("titanic")

filtered_df = df.groupby("pclass").filter(lambda x: len(x) >= 200)

print(filtered_df["pclass"].value_counts())

print("Passengers remaining:", len(filtered_df))

pclass
3    491
1    216
Name: count, dtype: int64
Passengers remaining: 707


### t8.py

In [15]:
import seaborn as sns

df = sns.load_dataset("titanic")

port_stats = df.groupby("embarked").agg(
    total_passengers=("survived", "count"),
    survival_rate=("survived", "mean"),
    average_fare=("fare", "mean")
)

port_stats = port_stats.sort_values("survival_rate", ascending=False)

print(port_stats)

          total_passengers  survival_rate  average_fare
embarked                                               
C                      168       0.553571     59.954144
Q                       77       0.389610     13.276030
S                      644       0.336957     27.079812


### t9.py

In [16]:
import pandas as pd

students = pd.DataFrame({
    "id": [1, 2, 3, 4, 5],
    "name": ["Aryan", "Mihir", "Vidhan", "Harsh", "Dwip"],
    "age": [18, 19, 20, 18, 21]
})

grades = pd.DataFrame({
    "id": [1, 2, 2, 3, 6],
    "subject": ["Math", "Math", "Science", "Math", "Math"],
    "marks": [85, 90, 88, 75, 95]
})

inner_join = pd.merge(students, grades, on="id", how="inner")
left_join = pd.merge(students, grades, on="id", how="left")
right_join = pd.merge(students, grades, on="id", how="right")
outer_join = pd.merge(students, grades, on="id", how="outer")

print("Inner Join")
print(inner_join)

print("\nLeft Join")
print(left_join)

print("\nRight Join")
print(right_join)

print("\nOuter Join")
print(outer_join)

Inner Join
   id    name  age  subject  marks
0   1   Aryan   18     Math     85
1   2   Mihir   19     Math     90
2   2   Mihir   19  Science     88
3   3  Vidhan   20     Math     75

Left Join
   id    name  age  subject  marks
0   1   Aryan   18     Math   85.0
1   2   Mihir   19     Math   90.0
2   2   Mihir   19  Science   88.0
3   3  Vidhan   20     Math   75.0
4   4   Harsh   18      NaN    NaN
5   5    Dwip   21      NaN    NaN

Right Join
   id    name   age  subject  marks
0   1   Aryan  18.0     Math     85
1   2   Mihir  19.0     Math     90
2   2   Mihir  19.0  Science     88
3   3  Vidhan  20.0     Math     75
4   6     NaN   NaN     Math     95

Outer Join
   id    name   age  subject  marks
0   1   Aryan  18.0     Math   85.0
1   2   Mihir  19.0     Math   90.0
2   2   Mihir  19.0  Science   88.0
3   3  Vidhan  20.0     Math   75.0
4   4   Harsh  18.0      NaN    NaN
5   5    Dwip  21.0      NaN    NaN
6   6     NaN   NaN     Math   95.0


### t10.py

In [17]:
import pandas as pd

jan_sales = pd.DataFrame({
    "Product": ["A", "B", "C"],
    "Sales": [1200, 1500, 1800]
})

feb_sales = pd.DataFrame({
    "Product": ["A", "B", "C"],
    "Sales": [1300, 1400, 1900]
})

jan_sales["Month"] = "January"
feb_sales["Month"] = "February"

yearly_sales = pd.concat([jan_sales, feb_sales], ignore_index=True)

print(yearly_sales)

  Product  Sales     Month
0       A   1200   January
1       B   1500   January
2       C   1800   January
3       A   1300  February
4       B   1400  February
5       C   1900  February


### t11.py

In [18]:
import pandas as pd
import seaborn as sns

df = sns.load_dataset("titanic")

class_labels = pd.DataFrame({
    "pclass": [1, 2, 3],
    "class_label": ["First", "Second", "Third"]
})

merged_df = pd.merge(df, class_labels, on="pclass", how="left")

print(merged_df[["pclass", "class_label"]].head())

print("\nAll rows have a label:",
      merged_df["class_label"].notna().all())

   pclass class_label
0       3       Third
1       1       First
2       3       Third
3       1       First
4       3       Third

All rows have a label: True


### t12.py

In [19]:
import pandas as pd

students = pd.DataFrame({
    "id": [1, 2, 3, 4],
    "name": ["Aryan", "Mihir", "Vidhan", "Harsh"]
})

grades = pd.DataFrame({
    "id": [1, 2, 2, 3],
    "marks": [85, 90, 90, 75]
})

merged_df = pd.merge(students, grades, on="id", how="inner")

duplicates = merged_df[merged_df.duplicated()]

print("Duplicate Rows:")
print(duplicates)

clean_df = merged_df.drop_duplicates()

print("\nAfter Removing Duplicates:")
print(clean_df)

Duplicate Rows:
   id   name  marks
2   2  Mihir     90

After Removing Duplicates:
   id    name  marks
0   1   Aryan     85
1   2   Mihir     90
3   3  Vidhan     75


### t13.py

In [20]:
import seaborn as sns

df = sns.load_dataset("titanic")

pt = df.pivot_table(
    values="age",
    index="pclass",
    columns="survived",
    aggfunc=["mean", "std"],
    margins=True
)

print(pt)

               mean                              std                      
survived          0          1        All          0          1        All
pclass                                                                    
1         43.695312  35.368197  38.233441  15.284243  13.760017  14.802856
2         33.544444  25.901566  29.877630  12.151581  14.837787  14.001077
3         26.555556  20.646118  25.140620  12.334882  11.995047  12.495398
All       30.626179  28.343690  29.699118  14.172110  14.950952  14.526497


### t14.py

In [21]:
import pandas as pd
import seaborn as sns

df = sns.load_dataset("titanic")

survival_pct = pd.crosstab(
    [df["sex"], df["embarked"]],
    df["survived"],
    normalize="index"
) * 100

print(survival_pct)

highest_group = survival_pct[1].idxmax()
highest_rate = survival_pct[1].max()

print("\nHighest Survival Group:")
print("Group:", highest_group)
print("Survival Rate:", highest_rate)

survived                 0          1
sex    embarked                      
female C         12.328767  87.671233
       Q         25.000000  75.000000
       S         31.034483  68.965517
male   C         69.473684  30.526316
       Q         92.682927   7.317073
       S         82.539683  17.460317

Highest Survival Group:
Group: ('female', 'C')
Survival Rate: 87.67123287671232


### t15.py

In [22]:
import pandas as pd
import seaborn as sns

df = sns.load_dataset("titanic")

fare_table = pd.pivot_table(
    df,
    values="fare",
    index="pclass",
    columns="embarked",
    aggfunc="sum",
    fill_value=0
)

print(fare_table)

top_port = fare_table.idxmax(axis=1)

print("\nPort with Highest Revenue in Each Class:")
print(top_port)

embarked          C         Q          S
pclass                                  
1         8901.0750  180.0000  8936.3375
2          431.0917   37.0500  3333.7000
3          740.1295  805.2043  5169.3613

Port with Highest Revenue in Each Class:
pclass
1    S
2    S
3    S
dtype: object


### t16.py

In [23]:
import seaborn as sns

df = sns.load_dataset("titanic")

port_names = {
    "C": "Cherbourg",
    "Q": "Queenstown",
    "S": "Southampton"
}

df["embarked"] = df["embarked"].map(port_names)

print(df[["embarked"]].head(10))

      embarked
0  Southampton
1    Cherbourg
2  Southampton
3  Southampton
4  Southampton
5   Queenstown
6  Southampton
7  Southampton
8  Southampton
9    Cherbourg


### t17.py

In [24]:
import seaborn as sns

df = sns.load_dataset("titanic")

def get_outcome(row):
    if row["survived"] == 1 and row["fare"] > 50:
        return "Survived-Rich"
    elif row["survived"] == 1 and row["fare"] <= 50:
        return "Survived-Poor"
    elif row["survived"] == 0 and row["fare"] > 50:
        return "Died-Rich"
    else:
        return "Died-Poor"

df["Outcome"] = df.apply(get_outcome, axis=1)

print(df[["survived", "fare", "Outcome"]].head())

   survived     fare        Outcome
0         0   7.2500      Died-Poor
1         1  71.2833  Survived-Rich
2         1   7.9250  Survived-Poor
3         1  53.1000  Survived-Rich
4         0   8.0500      Died-Poor


### t18.py

In [25]:
import pandas as pd
import seaborn as sns

df = sns.load_dataset("titanic")

df["age_cut"] = pd.cut(df["age"], bins=5)

df["age_qcut"] = pd.qcut(df["age"], q=5)

print("Distribution using cut:")
print(df["age_cut"].value_counts().sort_index())

print("\nDistribution using qcut:")
print(df["age_qcut"].value_counts().sort_index())

Distribution using cut:
age_cut
(0.34, 16.336]      100
(16.336, 32.252]    346
(32.252, 48.168]    188
(48.168, 64.084]     69
(64.084, 80.0]       11
Name: count, dtype: int64

Distribution using qcut:
age_qcut
(0.419, 19.0]    164
(19.0, 25.0]     137
(25.0, 31.8]     127
(31.8, 41.0]     144
(41.0, 80.0]     142
Name: count, dtype: int64


### t19.py

In [26]:
import pandas as pd

df = pd.DataFrame({
    "A": [1.23456, 2.34567, 3.45678],
    "B": [4.56789, 5.67891, 6.78912]
})

df = df.applymap(lambda x: round(x, 2))

print(df)

      A     B
0  1.23  4.57
1  2.35  5.68
2  3.46  6.79


C:\Users\rathi\AppData\Local\Temp\ipykernel_19176\1230071647.py:8: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: round(x, 2))


### t20.py

In [27]:
import pandas as pd

df = pd.read_csv("student.csv")

df["name"] = df["name"].str.strip().str.title()

df["first_name"] = df["name"].str.split().str[0]
df["last_name"] = df["name"].str.split().str[-1]

print(df)

            name first_name last_name
0    Mihir Rathi      Mihir     Rathi
1  Pratham Patel    Pratham     Patel
2     Akash Shah      Akash      Shah


### t21.py

In [28]:
import pandas as pd

df = pd.DataFrame({
    "product_code": ["AB-1234-XY", "CD-5678-ZZ", "EF-9012-AA"]
})

df["category"] = df["product_code"].str.extract(r"^([A-Z]{2})")
df["ID"] = df["product_code"].str.extract(r"-(\d{4})-")
df["suffix"] = df["product_code"].str.extract(r"([A-Z]{2})$")

print(df)

  product_code category    ID suffix
0   AB-1234-XY       AB  1234     XY
1   CD-5678-ZZ       CD  5678     ZZ
2   EF-9012-AA       EF  9012     AA


### t22.py

In [29]:
import seaborn as sns

df = sns.load_dataset("titanic")

print(df["who"].value_counts()[["man", "woman"]])

who
man      537
woman    271
Name: count, dtype: int64


### t23.py

In [30]:
import pandas as pd

df = pd.DataFrame({
    "email": [
        "mihir@gmail.com",
        "akash@yahoo.com",
        "pratham@outlook.com"
    ]
})

df["domain"] = df["email"].str.extract(r'@([^.]+)')

print(df)

                 email   domain
0      mihir@gmail.com    gmail
1      akash@yahoo.com    yahoo
2  pratham@outlook.com  outlook


### t24.py

In [31]:
import pandas as pd
import seaborn as sns

df = sns.load_dataset("titanic")

df = df.drop_duplicates()

df["age"] = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df["fare"] = df["fare"].fillna(df["fare"].median())

df["FamilySize"] = df["sibsp"] + df["parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

df["AgeGroup"] = pd.cut(
    df["age"],
    bins=[0, 12, 18, 35, 60, 100],
    labels=["Child", "Teen", "YoungAdult", "Adult", "Senior"]
)

df["FareGroup"] = pd.qcut(df["fare"], 4, labels=["Low", "Medium", "High", "VeryHigh"])

stats = df.groupby(["sex", "pclass"]).agg(
    passengers=("survived", "count"),
    survival_rate=("survived", "mean"),
    avg_age=("age", "mean"),
    avg_fare=("fare", "mean")
).reset_index()

stats["survival_rate"] = stats["survival_rate"] * 100

df.to_csv("titanic_clean.csv", index=False)

with pd.ExcelWriter("titanic_report.xlsx") as writer:
    df.to_excel(writer, sheet_name="Data", index=False)
    stats.to_excel(writer, sheet_name="Stats", index=False)

print("Clean CSV exported: titanic_clean.csv")
print("Excel report exported: titanic_report.xlsx")
print("\nGroupby Statistics:")
print(stats)

Clean CSV exported: titanic_clean.csv
Excel report exported: titanic_report.xlsx

Groupby Statistics:
      sex  pclass  passengers  survival_rate    avg_age    avg_fare
0  female       1          93      96.774194  34.110215  106.521774
1  female       2          73      91.780822  28.424658   22.194921
2  female       3         127      47.244094  23.246063   15.704004
3    male       1         121      37.190083  39.127438   67.552617
4    male       2          92      18.478261  30.960109   21.550136
5    male       3         278      15.827338  26.847734   12.720727


### t25.py

In [32]:
import seaborn as sns

df = sns.load_dataset("titanic")

for pclass in df["pclass"].unique():
    class_df = df[df["pclass"] == pclass]
    class_df.to_csv(f"titanic_class_{pclass}.csv", index=False)

print("CSV files saved successfully.")

CSV files saved successfully.
